# Préparer lh_source

Notebook de préparation Contoso, à exécuter dans Microsoft Fabric avec **PySpark**. Attachez **lh_source** comme lakehouse par défaut, avec les schémas activés. Les cellules 2 à 4 remplacent uniquement les tables de démonstration `dbo.sites`, `dbo.emission_factors`, `dbo.consumption` et `dbo.consumption_latest_day`. N'utilisez pas un lakehouse de production.

Exécuter tout initialise l'état **before** ; la bascule **after**, en cellule 6, est désactivée par défaut. Les données sont publiques et synthétiques, les facteurs fictifs. Aucun secret ni identifiant de tenant n'est requis dans ce document. Le téléchargement réseau et les écritures Delta doivent être testés dans votre tenant.

In [ ]:
import csv
import io
from datetime import date
from decimal import Decimal, InvalidOperation
from urllib.request import urlopen

import notebookutils
from pyspark.sql import functions as functions
from pyspark.sql.types import DateType, DecimalType, IntegerType, StringType, StructField, StructType

context = notebookutils.runtime.context
if context.get('defaultLakehouseName') != 'lh_source':
    raise RuntimeError('Attachez lh_source comme lakehouse par défaut, puis redémarrez la session Spark si nécessaire.')
base_url = 'https://raw.githubusercontent.com/AmineLemsih/hands-on-lab-microsoft-fabric-end-to-end/main/data/csv/'

def download_rows(filename, expected_columns):
    with urlopen(base_url + filename, timeout=60) as response:
        content = response.read().decode('utf-8-sig')
    reader = csv.DictReader(io.StringIO(content))
    if reader.fieldnames != expected_columns:
        raise ValueError('Schéma inattendu pour ' + filename)
    return list(reader)

consumption_columns = ['site_id', 'date', 'year', 'kwh_elec', 'kwh_gas', 'avg_temp']
consumption_schema = StructType([
    StructField('site_id', StringType(), False),
    StructField('date', DateType(), False),
    StructField('year', IntegerType(), False),
    StructField('kwh_elec', DecimalType(18, 2), False),
    StructField('kwh_gas', DecimalType(18, 2), False),
    StructField('avg_temp', DecimalType(18, 2), False),
])

def parse_consumption(rows):
    parsed, rejected = [], 0
    for row in rows:
        try:
            values = [Decimal(row[column]) for column in consumption_columns[3:]]
            if not row['site_id'] or not all(value.is_finite() for value in values):
                raise ValueError('Observation invalide')
            parsed.append((row['site_id'], date.fromisoformat(row['date']), int(row['year']), *values))
        except (ValueError, InvalidOperation):
            rejected += 1
    return parsed, rejected

def save_demo_table(frame, table_name, expected_count):
    if context.get('defaultLakehouseName') != 'lh_source':
        raise RuntimeError('Lakehouse par défaut incorrect')
    if table_name not in {'sites', 'emission_factors', 'consumption', 'consumption_latest_day'}:
        raise ValueError('Table hors du périmètre du lab')
    if frame.count() != expected_count:
        raise ValueError('Volume inattendu pour ' + table_name)
    target = 'dbo.' + table_name
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(target)
    if spark.table(target).count() != expected_count:
        raise RuntimeError('Volume après écriture incorrect pour ' + target)
    print(target + ' : ' + str(expected_count) + ' lignes')

In [ ]:
site_rows = download_rows('sites.csv', ['site_id', 'site_name', 'region', 'activity', 'area_m2', 'opening_date'])
factor_rows = download_rows('emission_factors.csv', ['year', 'elec_kgco2e_per_kwh', 'gas_kgco2e_per_kwh', 'source'])
annual_rows = download_rows('consumption_2025.csv', consumption_columns)
latest_rows = download_rows('consumption_latest_day_before.csv', consumption_columns)

site_schema = StructType([
    StructField('site_id', StringType(), False), StructField('site_name', StringType(), False),
    StructField('region', StringType(), False), StructField('activity', StringType(), False),
    StructField('area_m2', IntegerType(), False), StructField('opening_date', DateType(), False),
])
factor_schema = StructType([
    StructField('year', IntegerType(), False),
    StructField('elec_kgco2e_per_kwh', DecimalType(12, 6), False),
    StructField('gas_kgco2e_per_kwh', DecimalType(12, 6), False),
    StructField('source', StringType(), False),
])
site_values = [(row['site_id'], row['site_name'], row['region'], row['activity'], int(row['area_m2']), date.fromisoformat(row['opening_date'])) for row in site_rows]
factor_values = [(int(row['year']), Decimal(row['elec_kgco2e_per_kwh']), Decimal(row['gas_kgco2e_per_kwh']), row['source']) for row in factor_rows]
annual_values, rejected = parse_consumption(annual_rows)
latest_values, latest_rejected = parse_consumption(latest_rows)
assert len(annual_rows) == 10950 and rejected == 110 and len(annual_values) == 10840
assert len(site_values) == 30 and len({row[0] for row in site_values}) == 30
assert len(factor_values) == 1 and factor_values[0][0] == 2025
assert len(latest_values) == 30 and latest_rejected == 0
assert len({(row[0], row[1]) for row in annual_values}) == 10840
assert {row[0] for row in latest_values} == {row[0] for row in site_values}

sites_frame = spark.createDataFrame(site_values, site_schema)
factors_frame = spark.createDataFrame(factor_values, factor_schema)
annual_frame = spark.createDataFrame(annual_values, consumption_schema).withColumn('month_start', functions.trunc('date', 'month'))
latest_frame = spark.createDataFrame(latest_values, consumption_schema)
regional_before = latest_frame.join(sites_frame, 'site_id').groupBy('region').agg(functions.sum(functions.col('kwh_elec') + functions.col('kwh_gas')).alias('total_kwh'))
assert regional_before.filter(functions.col('total_kwh') >= 10000).count() == 0

save_demo_table(sites_frame, 'sites', 30)
save_demo_table(factors_frame, 'emission_factors', 1)
save_demo_table(annual_frame, 'consumption', 10840)
save_demo_table(latest_frame, 'consumption_latest_day', 30)
print('Initialisation before terminée. Historique nettoyé : 110 observations rejetées, sans imputation.')

In [ ]:
summary = spark.table('dbo.consumption').agg(
    functions.sum('kwh_elec').alias('electricity_kwh'),
    functions.sum('kwh_gas').alias('gas_kwh'),
).first()
assert summary['electricity_kwh'] == Decimal('2896164.51')
assert summary['gas_kwh'] == Decimal('1686455.14')
display(regional_before.orderBy('region'))
print('Contrôles réussis. Attendre la synchronisation SQL, puis créer ou actualiser sm_energy_report.')

## Bascule after, uniquement pendant le Lab 5

Attendez que les règles Activator aient observé l'état sous 10 000 kWh. Dans la cellule suivante, passez `apply_after` à `True`, puis exécutez **cette cellule seulement**. Elle remplace uniquement `consumption_latest_day`. Remettez ensuite `apply_after` à `False` et actualisez **sm_energy_report** dans Power BI. La Bretagne doit passer de 2 724,55 à 36 724,55 kWh. Pour revenir à before, réexécutez les cellules 2 à 4. Ne relancez pas l'initialisation entre le franchissement et sa détection. Une session Spark redémarrée exige de réexécuter la cellule 2 avant la bascule.

In [ ]:
apply_after = False

if apply_after:
    after_rows = download_rows('consumption_latest_day_after.csv', consumption_columns)
    after_values, after_rejected = parse_consumption(after_rows)
    assert len(after_values) == 30 and after_rejected == 0
    after_frame = spark.createDataFrame(after_values, consumption_schema)
    assert after_frame.select('site_id').distinct().count() == 30
    regional_after = after_frame.join(spark.table('dbo.sites'), 'site_id').groupBy('region').agg(
        functions.sum(functions.col('kwh_elec') + functions.col('kwh_gas')).alias('total_kwh')
    )
    exceeded = regional_after.filter(functions.col('total_kwh') > 10000).collect()
    assert len(exceeded) == 1 and exceeded[0]['region'] == 'Bretagne'
    assert exceeded[0]['total_kwh'] == Decimal('36724.55')
    save_demo_table(after_frame, 'consumption_latest_day', 30)
    display(regional_after.orderBy('region'))
    print('État after chargé. Actualisez sm_energy_report, puis mesurez la latence Teams.')
else:
    print('Bascule ignorée : apply_after=False. L’état before est conservé après Exécuter tout.')